# E10 — Frozen Prospective 36 × 3 Intervention Evaluation

**Primary prospective GPU experiment.** 36 untouched-by-source-registry problems (12 Arithmetic / 12 Countdown / 12 Multistep), each with one fixed initial generation plus **three precommitted actions** = **144 total calls (36 + 108)**. The frozen E06 router is applied after results, not refitted. This notebook embeds the original E06 manifest, bank and policy and requires no previous result ZIP.

**Protocol amendment, made before any E10 runs:** the earlier E06 generic specification listed 96 action tokens, while its winning finalization policy used 128. Here finalization and full regeneration each use 128 tokens, while selective repair regenerates the remaining suffix (at most 96) with entropy decoding. **This is not compute matched**; forward-token passes and time are reported for all arms. Do not mix this amended test with an unamended 96-budget protocol.

A Colab disconnect cannot be prevented. Records are saved immediately to `MyDrive/RepairScopeX/E10_Prospective_v2/records`. A single Run All will run up to 12 new calls or 29 minutes; repeat **this same notebook only** to resume. GPU execution has not been verified locally.

In [ ]:

import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX_E10E11'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned ALoDLM source:',CODE_COMMIT)


In [ ]:
import json,os,re,time,random,math,ast,hashlib,zipfile,zlib,base64,gc,csv
from pathlib import Path
from collections import Counter,defaultdict
from types import SimpleNamespace
import numpy as np,torch,pandas as pd
from huggingface_hub import snapshot_download
from google.colab import drive,files
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('GPU runtime required (T4 or better)')
if torch.cuda.get_device_properties(0).total_memory<12*2**30: raise RuntimeError('GPU must have >=12GiB VRAM')
drive.mount('/content/drive')
STUDY='RSX_E10_Prospective_v2'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/E10_Prospective_v2')
RECS=ROOT/'records';RECS.mkdir(parents=True,exist_ok=True)
DATA=json.loads(zlib.decompress(base64.b64decode('...')))
MODEL_ID='amazon/ALoDLM-1.7B'
MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
Q=0.4;TAU=0.4;WINDOW=16
PRIOR_PROMPT='\n\nGive a short solution. End with one line: FINAL_ANSWER: <number>. For Countdown include an expression using all inputs once.'
DIRECT_PROMPT='\n\nDo not provide working. Reply with one line containing the final numerical answer. For Countdown provide a valid expression before the final number.'
FRESH_PROMPT='\n\nIndependently solve the problem from the beginning. Verify any computation. End with FINAL_ANSWER: <number>. For Countdown give a valid expression using each number once.'
def digest(v):
    return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def save_new_json(path,obj):
    if path.exists():
        old=json.loads(path.read_text());assert old==obj,'Immutable protocol/data mismatch at '+str(path)
    else:
        temp=path.with_name(path.name+'.pending');temp.write_text(json.dumps(obj,indent=2,sort_keys=True,ensure_ascii=False));os.replace(temp,path)
def atomic_record(obj):
    assert obj['protocol_hash']==PROTOCOL_HASH
    tid=obj['trial_id'];p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if p.exists():assert json.loads(p.read_text())==obj,'Conflicting trial ID '+tid
    else:
        t=p.with_suffix('.pending');t.write_text(json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False));os.replace(t,p)
def existing(tid):
    p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not p.exists():return None
    obj=json.loads(p.read_text());assert obj['trial_id']==tid and obj['protocol_hash']==PROTOCOL_HASH
    return obj
def read_records():
    out=[json.loads(f.read_text()) for f in sorted(RECS.glob('*.json'))]
    assert len(out)==len({r['trial_id'] for r in out})
    assert all(r['protocol_hash']==PROTOCOL_HASH for r in out)
    return out
def set_seed(seed):
    random.seed(seed);np.random.seed(seed%(2**32-1));torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def make_prompt(content):
    return list(tokenizer.apply_chat_template([{'role':'user','content':content}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def render(ids):return tokenizer.decode(list(map(int,ids)),skip_special_tokens=True).strip()
def model_call(prompt,prefix,mode,max_new,seed):
    set_seed(seed)
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(max_new),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats();start=time.monotonic()
    out=decoder.generate(make_prompt(prompt)+list(map(int,prefix)),cfg)
    new=list(map(int,out['token_ids']))
    return {'out':render(list(prefix)+new),'ids':list(prefix)+new,'new_ids':new,
            'generated_tokens':int(out.get('generated_tokens',len(new))),
            'loop_token_passes':int(out.get('loop_token_passes',0)),
            'outer_steps':int(out.get('outer_steps',0)),
            'stop_reason':str(out.get('stop_reason','unknown')),
            'wall_seconds':round(time.monotonic()-start,3),
            'peak_vram_gb':round(float(torch.cuda.max_memory_allocated()/2**30),3)}
def seed_for(pid):return 602600 + int(hashlib.sha256(pid.encode()).hexdigest()[:6],16) % 999999

def character_cut(ids,offset):
    for i in range(1,len(ids)):
        if len(render(ids[:i]))>=offset:return max(1,i-1)
    return max(1,len(ids)//2)
def repair_cut(ids,action):
    n=len(ids)
    if n<2:return 0
    if action=='first_math':
        m=re.search(r'(?:\d+\s*[+*/-]\s*\d+)|(?:\d+\s*=)',render(ids))
        return min(n-1,max(1,character_cut(ids,m.start()) if m else n//2))
    if action=='tail50':return min(n-1,max(1,n//2))
    raise ValueError(action)
def snapshot_export():
    archive=Path('/content')/(STUDY+'_Checkpoint.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob('*')):
            if p.is_file() and not p.name.endswith('.pending'):z.write(p,p.relative_to(ROOT))
    print('Archive',archive,'size',archive.stat().st_size,'saved records',len(read_records()))
    try:files.download(str(archive))
    except Exception as exc:print('Auto-download failed; file remains at',archive,repr(exc))
    return archive


In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')

def strict_completed_correct(text,p):
    ev=grade_output(str(text).replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]',''),p)
    if p['kind']!='countdown':
        v,orig=last_explicit_number(str(text))
        return bool(closenum(v,p['target']) and orig!='conflicting_markers')
    if ev['valid_expression'] is None: return False
    # Accept only complete final-answer lines, NOT incomplete math reasoning.
    for match in re.findall(r'(?im)^\s*FINAL_ANSWER\s*:\s*(.*?)\s*$',str(text)):
        raw=match.strip().replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]','')
        try:
            if closenum(float(raw),p['target']):return True
        except (ValueError,TypeError):pass
        exp=safe_expression(raw)
        if exp and closenum(exp['value'],p['target']) and sorted(exp['literals'])==sorted(map(float,p['numbers'])):return True
    return False

def dual_score(text,p):
    main=grade_output(text,p)
    complete=strict_completed_correct(text,p)
    return {'primary':main,'complete_correct':bool(complete),
            'grading_disagreement':bool(main['success'])!=bool(complete),
            'class':('correct' if complete else ('definite_wrong' if main['eligible_semantic_failure'] else 'ambiguous'))}
# Scoring unit tests against prior observed failure modes.
test_cd={'kind':'countdown','target':14,'numbers':[2,3,4]}
assert strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: 14'.replace('\\n','\n'),test_cd)
assert not strict_completed_correct('2 + 3 * 4 = 14',test_cd)
assert not strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: \\('.replace('\\n','\n'),test_cd)


In [ ]:

BANK={x['problem_id']:x for x in DATA['bank']}
MANIFEST=DATA['manifest']; POLICY=DATA['policy'];SPEC=DATA['spec']
assert len(MANIFEST['problems'])==36 and len({x['problem_id'] for x in MANIFEST['problems']})==36
assert SPEC['manifest_sha256']==digest(MANIFEST) and SPEC['frozen_policy_sha256']==digest(POLICY)
TOUCHED=set()
for ids in DATA['prior_touched'].values():TOUCHED.update(ids)
assert not TOUCHED & {x['problem_id'] for x in MANIFEST['problems']},'Historical ID leakage detected!'
for entry in MANIFEST['problems']:
    p=BANK[entry['problem_id']]
    assert p['split']=='heldout' and p['kind']==entry['kind'] and p['problem_fingerprint']==entry['problem_fingerprint']
    assert hashlib.sha256(p['prompt'].encode()).hexdigest()==entry['prompt_sha256']
PLAN=[(e['problem_id'],phase) for e in MANIFEST['problems'] for phase in ['source','finalize','selective_remask','regenerate']]
assert len(PLAN)==144
PROTOCOL={'spec':SPEC,'manifest':MANIFEST,'policy':POLICY,'code_commit':CODE_COMMIT,'model_rev':MODEL_REV,
          'q':Q,'tau':TAU,'window':WINDOW,'plan':PLAN}
PROTOCOL_HASH=digest(PROTOCOL)
save_new_json(ROOT/'protocol.json',{'protocol_hash':PROTOCOL_HASH,'protocol':PROTOCOL})
save_new_json(ROOT/'E06_original_manifest.json',MANIFEST)
save_new_json(ROOT/'E06_original_frozen_policy.json',POLICY)
print('36 sources and 108 action outcomes planned. Existing:',len(read_records()))


In [ ]:

sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
gc.collect();torch.cuda.empty_cache()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);tokenizer=decoder.tokenizer
assert next(decoder.model.parameters()).dtype==torch.float32
print('Pinned FP32 ALoDLM loaded')


In [ ]:

SESSION_MAX_NEW=12
SESSION_MAX_MINUTES=29
started=time.monotonic();new=0;failures=[]
for pid,phase in PLAN:
    if new>=SESSION_MAX_NEW or time.monotonic()-started>=SESSION_MAX_MINUTES*60:break
    tid=f'{STUDY}|{pid}|{phase}'
    if existing(tid) is not None:continue
    p=BANK[pid];base=seed_for(pid);src=existing(f'{STUDY}|{pid}|source')
    if phase!='source' and src is None:raise RuntimeError('Source missing before action '+pid)
    try:
        if phase=='source':
            res=model_call(p['prompt']+PRIOR_PROMPT,[],mode='left1',max_new=96,seed=base)
            rec={'trial_id':tid,'phase':phase,'source_trial_id':None,'problem_id':pid,'split':'heldout','kind':p['kind'],
                 'action':'source','source_output':None,'output':res['out'],'token_ids':res['ids'],
                 'stop_reason':res['stop_reason'],'generated_tokens':res['generated_tokens'],
                 'loop_token_passes':res['loop_token_passes'],'wall_seconds':res['wall_seconds'],
                 'peak_vram_gb':res['peak_vram_gb'],'score':dual_score(res['out'],p),
                 'protocol_hash':PROTOCOL_HASH,'seed':base,'budget':96,'decoder':'left1','preserved_tokens':0}
        else:
            tokens=src['token_ids'];cut=0
            if phase=='finalize':
                res=model_call(p['prompt']+DIRECT_PROMPT,[],mode='left1',max_new=128,seed=base)
                mode='left1';requested=128
            elif phase=='regenerate':
                res=model_call(p['prompt']+FRESH_PROMPT,[],mode='left1',max_new=128,seed=base)
                mode='left1';requested=128
            else:
                mode='entropy'
                if len(tokens)<2:
                    # Do not silently switch actions: record an explicit non-executable no-op.
                    res={'out':src['output'],'ids':tokens,'new_ids':[],'generated_tokens':0,
                         'loop_token_passes':0,'wall_seconds':0.,'peak_vram_gb':0.,
                         'stop_reason':'non_executable_short_source'}
                    requested=0
                else:
                    cut=repair_cut(tokens,'first_math');requested=min(96,max(1,len(tokens)-cut))
                    res=model_call(p['prompt']+PRIOR_PROMPT,tokens[:cut],mode='entropy',max_new=requested,seed=base)
                    assert res['ids'][:cut]==tokens[:cut]
            rec={'trial_id':tid,'phase':phase,'action':phase,'source_trial_id':src['trial_id'],'source_output':src['output'],
                 'source_stop_reason':src['stop_reason'],'problem_id':pid,'split':'heldout','kind':p['kind'],
                 'output':res['out'],'stop_reason':res['stop_reason'],'generated_tokens':res['generated_tokens'],
                 'loop_token_passes':res['loop_token_passes'],'wall_seconds':res['wall_seconds'],
                 'peak_vram_gb':res['peak_vram_gb'],'score':dual_score(res['out'],p),
                 'protocol_hash':PROTOCOL_HASH,'seed':base,'budget':requested,'decoder':mode,
                 'preserved_tokens':cut,'initial_correct':bool(src['score']['complete_correct'])}
        atomic_record(rec);new+=1
        print('NEW',new,phase,pid,rec['score']['class'],flush=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache();failures.append({'trial_id':tid,'error':'GPU_OOM'});break
    except Exception as exc:
        failures.append({'trial_id':tid,'error':repr(exc)});break
if failures:
    (ROOT/'last_failure.json').write_text(json.dumps(failures,indent=2))
    print('STOPPED due to:',failures[-1])
else:
    print('Session limit reached or all planned records saved.')
print('Saved records',len(read_records()),'/',len(PLAN))


In [ ]:

recs=read_records();by={(r['problem_id'],r['phase']):r for r in recs}
complete=[p for p in MANIFEST['problems'] if all((p['problem_id'],a) in by for a in ('source','finalize','selective_remask','regenerate'))]
rows=[];checks=[]
for e in MANIFEST['problems']:
    pid=e['problem_id'];src=by.get((pid,'source'))
    if src is None:continue
    selected='selective_remask' if re.search(r'(?im)^\s*(?:FINAL_ANSWER|ANSWER)\s*:\s*[-+]?\d|\\boxed\s*\{\s*[-+]?\d',src['output']) else 'finalize'
    for action in ('source','finalize','selective_remask','regenerate'):
        r=by.get((pid,action))
        if r is None:continue
        checks.append({'problem_id':pid,'action':action,'recorded_success':int(r['score']['complete_correct']),
          'primary_success':int(r['score']['primary']['success']),'grader_disagreement':int(r['score']['grading_disagreement']),
          'generated_tokens':r['generated_tokens'],'pass_cost':r['loop_token_passes'],'wall_seconds':r['wall_seconds']})
    if all((pid,a) in by for a in ('finalize','selective_remask','regenerate')):
        rr={'problem_id':pid,'kind':e['kind'],'router_action':selected,'source_correct':int(src['score']['complete_correct'])}
        for action in ('finalize','selective_remask','regenerate'):
            r=by[(pid,action)];rr[action+'_correct']=int(r['score']['complete_correct'])
            rr[action+'_cost']=r['loop_token_passes']+src['loop_token_passes']
        rr['router_correct']=rr[selected+'_correct'];rr['router_cost']=rr[selected+'_cost']
        rows.append(rr)
if checks:pd.DataFrame(checks).to_csv(ROOT/'trial_audit.csv',index=False)
if rows:
    df=pd.DataFrame(rows);df.to_csv(ROOT/'paired_summary.csv',index=False)
    display(df.groupby('kind')[['router_correct','finalize_correct','selective_remask_correct','regenerate_correct','source_correct']].agg(['mean','size']))
status={'planned_sources':36,'planned_actions':108,'completed_records':len(recs),
        'completed_problem_quads':len(complete),'prospective_results_complete':len(complete)==36,
        'unique_problem_ids':len({r['problem_id'] for r in recs}),
        'frozen_router_unchanged':True,'manifest_sha256':SPEC['manifest_sha256'],
        'protocol_hash':PROTOCOL_HASH,
        'note':'If incomplete, treat comparisons as interim diagnostics only; never tune or select based on them.'}
save_new_json(ROOT/'status_snapshot.json',status) if not (ROOT/'status_snapshot.json').exists() else (ROOT/'status_snapshot.json').write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))
snapshot_export()
